### Practice: Large Language Models and Their Implications
<!-- ![img](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F4470ce74-e595-4750-92a5-5f21f040df6d_577x432.jpeg) -->
![img](https://i.imgur.com/QGYa2J8.jpeg)

In this notebook, you're gonna play with some of the largest language models on the Internet.

_Based on works of: Tim Dettmers, Ruslan Svirschevsky, Artem Chumachenko, Younes Belkada, Felix Marty, Yulian Gilyazev, Gosha Zolotov, Andrey Ishutin,  Elena Volf, Artemiy Vishnyakov, Svetlana Shirokovskih.

### Part 1: prompt engineering (4 points total)

In the assignment, we'll use public APIs that host the 100B+ models for inference. Your task is to prompt-engineer the model into solving a few tasks for you.


__Which API?__ You are free to use any publicly available API for general LM -- as long as it's __not a chat assistant__. So, gpt 3.5 is fine, but chatGPT is not. Here's a few options:

- BLOOM API - [bigscience/bloom](https://huggingface.co/bigscience/bloom) (on the right; recommended)
- OpenAI API (via VPN) - [openai.com/api](https://openai.com/api/)
- AI21 Jurrasic API - [ai21.com](https://www.ai21.com/blog/announcing-ai21-studio-and-jurassic-1)

These APIs may require you to create a (free) account on their platform. Please note that some APIs also have paid subscriptions. __You do not need to pay them__, this assignment was designed to be solved using free-tier subscriptions. If no APIs work for you, you can also solve these tasks with the 6.7B model that you will find later in this notebook - but this will make the tasks somewhat harder.

__Quests:__ you will need to solve 4 problems. For each one, please attach a short __description__ of your solution and a __screenshot__ from the API you use. _[If you use python APIs, show your python code with outputs]_

__Example:__ Tony is talking to Darth Vader ([BLOOM API](https://huggingface.co/bigscience/bloom)). Black text is written manually, blue text is generated.
<hr>

![img](https://i.imgur.com/a1QhKF7.png)
<hr>

__It is fine to roll back a few times,__ e.g. in the example above, the model first generated Vader lines twice in a row, and we rolled that back. However, if you need more than 1-2 rollbacks per session, you should probably try a different prompt.

__Task 1 (1 pt):__ arange a conversation between any two of the following:

- a celebrity or politician of your choice
- any fictional character (except Darth Vader)
- yourself

Compare two setups: a) you prompt with character names only b) you supply additional information (see example).

In [3]:
# Install required packages for local model inference
%pip install -q transformers torch accelerate


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
import os

login(token=os.getenv('HF_TOKEN'))

model_name = "bigscience/bloom-3b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def generate_text(prompt, max_new_tokens=150, temperature=0.8, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text[len(prompt):]

prompt = "The ancient forest whispered secrets to those who would listen,"
result = generate_text(prompt)
print(f"Prompt: {prompt}")
print(f"Generated: {result}")

Loading Llama 3 8B base model...


`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-10 11:12:04.322178: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-10 11:12:08.692442: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Model loaded on: cuda:0
Prompt: The ancient forest whispered secrets to those who would listen,
Generated:  and the people of the village would call upon the gods to protect them. The people were always careful to keep the forest intact, so that no one could steal the treasures hidden in the forest. One day, a great evil that would destroy the village came. The people were all scared, and they called upon the gods to save them. The gods answered the call, and they brought a great storm. The wind blew the trees down, and the entire village was buried under the rubble. The villagers were in such a terrible situation, and they couldn’t do anything. They tried to call upon the gods again, but they were unable to do anything. The villagers were so desperate that they decided to kill themselves. They told the gods


In [16]:
print("="*80)
print("SETUP A: Character names only (minimal context)")
print("="*80)

prompt_minimal = """Sherlock Holmes: Good evening, Professor Einstein.

Albert Einstein:"""

print("\nPrompt:")
print(prompt_minimal)
print("\n" + "-"*80)

response_a = generate_text(prompt_minimal, max_new_tokens=200, temperature=0.8)
print("Generated text:", response_a)

SETUP A: Character names only (minimal context)

Prompt:
Sherlock Holmes: Good evening, Professor Einstein.

Albert Einstein:

--------------------------------------------------------------------------------
Generated text:  Well, if you must call me Albert, then I must call you
Sherlock. I must call you Sherlock Holmes.

Sherlock Holmes: My dear fellow, you have a very interesting name.

Albert Einstein: You are quite right. But I do not pretend to be very
interesting.

Sherlock Holmes: Very well. I shall not waste time in trying to make you
interesting.

Albert Einstein: I am interested. You are very interesting, and I am very
interested. So that is settled. I shall call you Sherlock Holmes.

Sherlock Holmes: I shall call you Sherlock.

Albert Einstein: Well, if you must call me Albert, then I must call you
Sherlock.

Sherlock Holmes: My dear fellow, you have a very interesting name.

Albert Einstein: You are quite right. But I do not pretend to be very
interesting.

Sherlock Holmes:

In [18]:
print("="*80)
print("SETUP B: Character names with rich context")
print("="*80)

prompt_B = """The following is a conversation between Sherlock Holmes, the famous Victorian detective known for his exceptional powers of deduction and logical reasoning, and Albert Einstein, the brilliant theoretical physicist who developed the theory of relativity.


Sherlock Holmes:"""

print("\nPrompt B:")
print(prompt_B)
print("\n" + "-"*80)

response_b = generate_text(prompt_B, max_new_tokens=200, temperature=0.8)
print("Generated text:", response_b)


SETUP B: Character names with rich context

Prompt B:
The following is a conversation between Sherlock Holmes, the famous Victorian detective known for his exceptional powers of deduction and logical reasoning, and Albert Einstein, the brilliant theoretical physicist who developed the theory of relativity.


Sherlock Holmes:

--------------------------------------------------------------------------------
Generated text:  How do you know what is real and what isn’t?

Einstein: When you see it, of course.

Holmes: But you can see nothing.

Einstein: Yes. It’s the same thing.

Holmes: But you have to see it to know it, of course.

Einstein: Yes, I suppose so. But then again, you can see nothing.

Holmes: But you can hear nothing.

Einstein: Yes. It’s the same thing.

Holmes: But you can hear nothing.

Einstein: Yes. But then again, you can hear nothing.

Holmes: But you can smell nothing.

Einstein: Yes. But then again, you can smell nothing.

Holmes: But you can taste nothing.

Einstein

__Please choose task 2a or 2b (1pt)__ depending on your model (you can do both, but you will be awarded points for one of these two tasks).

__Task 2a: (for BLOOM or other multilingual model)__ zero-shot translation. Take the first verse of [Edgar Allan Poe's "Raven"](https://www.poetryfoundation.org/poems/48860/the-raven) and __translate it into French.__ (You are free to use any other text of at least the same size)

Original text: ```
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
```

Verify your translation by converting french back into english using a public machine translation service.

__Task 2b: (non-BLOOM):__ toxicity classification for [SetFit/toxic_conversations](https://huggingface.co/datasets/SetFit/toxic_conversations). Make the model solve binary classification (toxic vs not toxic) in the few shot mode. For few-shot examples, use 2-3 toxic and 2-3 non-toxic non-toxic examples. Measure accuracy on at least 25 samples. You may need to try several different prompts before you find the one that works.

In [ ]:
# Task 2a: Zero-shot translation (English to French)
raven_text = """Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
"'Tis some visitor," I muttered, "tapping at my chamber door—
            Only this and nothing more.\""""

translation_prompt = f"""Translate the following Edgar Allan Poe poem from English to French. 
Preserve archaic language, and Gothic atmosphere:

{raven_text}

Important: Maintain the original metaphors and specific vocabulary.
French translation:"""

french_translation = generate_text(translation_prompt, max_new_tokens=90, temperature=0.4)
print("French translation:")
print(french_translation)

French translation:


Un peu plus tard, un soir de minuit, alors que je méditais, fatigué et
déprimé, sur de nombreux volumes de livres oubliés—
    Quand je me suis endormi, presque endormi, soudain, une tapette
a sonné, tapant à ma porte.
«C'est un visiteur," j'ai murmuré, tapant à ma porte—
            Seulement cela et rien de plus.

I would


In [34]:
# Verify by translating back to English
backtranslation_prompt = f"""Translate the following French text to English:

{french_translation.strip()}

English translation:"""

english_back = generate_text(backtranslation_prompt, max_new_tokens=90, temperature=0.1)
print("\n" + "="*80)
print("Back-translation to English:")
print(english_back)


Back-translation to English:


Later, at midnight, while I meditated, tired and depressed, on many
books that had been forgotten—
    When I fell asleep, almost asleep, suddenly, a tap was heard,
tapping at my door.
"It's a visitor," I murmured, taping at my door—
            Only that and nothing more.

I would

French translation:

Un peu plus tard, au soir de minuit, alors que je méd



__Task 3 (1pt):__ create a prompt and few-shot examples tha make the model __change the gender pronouns__ of the main actor in a given sentence in any direction of your choice. E.g. the doctor took off _his_ mask <-> the doctor took of _her_ mask.


In [39]:
# Task 3: Gender pronoun conversion
gender_swap_prompt = """Change the gender pronouns in the following sentences:

Original: The doctor took off his mask and washed his hands.
Changed: The doctor took off her mask and washed her hands.

Original: The nurse adjusted her stethoscope before examining the patient.
Changed: The nurse adjusted his stethoscope before examining the patient.

Original: The pilot checked his instruments carefully.
Changed: The pilot checked her instruments carefully.

Original: The teacher gathered his books and left the classroom.
Changed:"""

result = generate_text(gender_swap_prompt, max_new_tokens=10, temperature=0.9)
print("Result:", result.strip())

Result: The teacher gathered her books and left the classroom.


__Task 4 (1pt):__ write a prompt and supply examples such that the model would __convert imperial units to metric units__ (miles -> kilometers; mph -> kph). More specifically, the model should rewrite a given sentence and replace all imperial units with their metric equivalents. After it works with basic distances and speed, try to find complicated examples where it does *not* work.

Please note that 1 mile is not equal to 1 km :)

In [43]:
# Task 4: Imperial to Metric conversion
conversion_prompt = """Convert imperial units to metric units in the following sentences:

Original: The speed limit is 60 miles per hour.
Converted: The speed limit is 97 kilometers per hour.

Original: He ran a distance of 5 miles in 30 minutes.
Converted: He ran a distance of 8 kilometers in 30 minutes.

Original: The car was traveling at 70 mph on the highway.
Converted: The car was traveling at 113 kph on the highway.

Original: The marathon is 26.2 miles long.
Converted:"""

result = generate_text(conversion_prompt, max_new_tokens=9, temperature=0.3)
print("Basic conversion result:", result.strip())

Basic conversion result: The marathon is 42.2 kilometers long.


### Part 2: local inference

Now, let's try and load the strongest model that can fit a typical Colab GPU (T4 with 16 GB as of spring 2023).

Our best candidates are the smaller versions of the best performing open source models:
- 7 Bn parameters version of [LLaMA](https://arxiv.org/pdf/2302.13971.pdf) - best for spring 2023, released by Facebook
- 7 Bn parameters version of [Falcon](https://falconllm.tii.ae) - close competitor to Llama, released in May 2023 by [Technology Innovation Institute of UAE](https://www.tii.ae).
- 6.7 Bn parameters version of [OPT](https://arxiv.org/abs/2205.01068) - top choice in this nomination in 2022, released by Facebook.

Beware: while these models are smaller than the ones in API, they're still over 60x larger than the BERT we played with last time. The code below will *just barely* fit into memory, so make sure you don't have anything else loaded. Sometimes you may need to restart runtime for the code to work.

It's a good time to restart your kernel and switch to GPU! (Runtime -> Change runtime type)
<center><img src="https://i.imgur.com/OOfDYzJ.png" width=240px></center>

In [5]:
#%pip uninstall -y bitsandbytes
%pip install --quiet bitsandbytes==0.40.2 torch==2.0.1 transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 auto-gptq==0.4.2
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
#import bitsandbytes as bnb
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [7]:
%pip install fsspec==2023.9.2 --quiet --force-reinstall

ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
model_name = 'TheBloke/Llama-2-13B-GPTQ'
import tempfile
import os

# loading Llama tokenizer ...
with tempfile.TemporaryDirectory() as temp_dir:
    tokenizer = transformers.AutoTokenizer.from_pretrained(
            model_name,
            cache_dir=temp_dir
        )
    tokenizer.pad_token_id = tokenizer.eos_token_id

    # ... and the model itself
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map='auto',
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        cache_dir=temp_dir
    )

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 00ce2655-88fc-40e0-9177-824ccb91404a)')' thrown while requesting HEAD https://huggingface.co/TheBloke/Llama-2-13B-GPTQ/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


ImportError: cannot import name 'maybe_sync' from 'fsspec.asyn' (/usr/local/lib/python3.10/dist-packages/fsspec/asyn.py)

## Text generation

**Comparison of strategies for language model text generation:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br> **Cons:** Computationally more complex and requires a good loss function. |

Documentation references:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

### Generation with HuggingFace

In [ ]:
prompt = 'The first discovered martian lifeform looks like'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=True, temperature=0.8)
# greedy inference:                                        do_sample=False)
# beam search for highest probability:                     num_beams=4)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

#### Low-level code for text generation

In [ ]:
prompt = "Moscow is the capital of"
# prompt = "Skippy, a young android, likes to dream about electric"

print(prompt, '\n')

voc = tokenizer.get_vocab()
voc_rev = {v:k for k, v in voc.items()}  # reverse vocab for decode

for i in range(10):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probs.flatten(), num_samples=1)

    next_token = tokenizer.decode(next_token_id)
    prompt += next_token

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    top_tokens = sorted_indices[:5]
    print(f"Step #{i} candidates:")
    for t, p in zip (top_tokens, sorted_probs):
        t = voc_rev[t.item()]
        print(f"{t:<10}: {p:.4f} ")

    print(f'\nChosen token: {next_token}', end='\n\n', flush=True)

Moscow is the capital of 

Step #0 candidates:
▁Russia   : 0.7616 
▁the      : 0.1795 
▁Russian  : 0.0218 
▁a        : 0.0058 
▁not      : 0.0022 

Chosen token: Russia

Step #1 candidates:
.         : 0.3238 
,         : 0.3188 
▁and      : 0.1845 
and       : 0.0554 
<0x0A>    : 0.0080 

Chosen token: ,

Step #2 candidates:
▁the      : 0.1961 
▁and      : 0.1857 
▁located  : 0.0688 
▁a        : 0.0603 
▁one      : 0.0562 

Chosen token: the

Step #3 candidates:
▁largest  : 0.4282 
▁most     : 0.1651 
▁country  : 0.0528 
▁biggest  : 0.0515 
▁world    : 0.0377 

Chosen token: world

Step #4 candidates:
'         : 0.4954 
’         : 0.3950 
s         : 0.0522 
▁largest  : 0.0054 
larg      : 0.0047 

Chosen token: '

Step #5 candidates:
s         : 0.9785 
sl        : 0.0057 
st        : 0.0023 
sf        : 0.0020 
ss        : 0.0016 

Chosen token: s

Step #6 candidates:
▁largest  : 0.8468 
▁biggest  : 0.0521 
▁most     : 0.0272 
larg      : 0.0189 
▁second   : 0.0142 

Chosen token:

**Task 5: write code for nucleus sampling generation (2 points)**:

Use the `nucleus_sampling()` template below. Look at the detailed generation code above for inspiration. __Please do not use model.generate__.

**Bonus task: write code for beam search (3 bonus points)**

In [ ]:
from typing import Tuple, List

def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> Tuple[str, List[str]]:
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(model.device)
    
    with torch.no_grad():
        logits = model.forward(**inputs).logits[0, -1, :]
    
    probs = torch.nn.functional.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    
    cumsum_probs = torch.cumsum(sorted_probs, dim=0)
    nucleus_mask = cumsum_probs <= prob
    nucleus_indices = sorted_indices[nucleus_mask]
    nucleus_probs = sorted_probs[nucleus_mask]
    
    if len(nucleus_indices) == 0:
        nucleus_indices = sorted_indices[:1]
        nucleus_probs = sorted_probs[:1]
    
    nucleus_probs = nucleus_probs / nucleus_probs.sum()
    sampled_idx = torch.multinomial(nucleus_probs, num_samples=1)
    sampled_token_id = nucleus_indices[sampled_idx]
    
    sampled_token = tokenizer.decode(sampled_token_id)
    possible_tokens = [tokenizer.decode(idx) for idx in nucleus_indices]
    
    return sampled_token, possible_tokens

In [ ]:
# Tests for nucleus sampling
test_prompt = "Elbrus is the highest"
next_token, possible_tokens = nucleus_sampling(model, tokenizer, test_prompt, prob=0.9)
print(test_prompt, next_token, possible_tokens)
assert next_token in possible_tokens
assert 3 <= len(possible_tokens) <= 3
assert sorted(possible_tokens) == ['mountain', 'peak', 'point']

test_prompt = "Large language models can learn to"
next_token, possible_tokens = nucleus_sampling(model, tokenizer, test_prompt, prob=0.4)
print(test_prompt, next_token, possible_tokens)
assert next_token in possible_tokens
assert sorted(possible_tokens) == ['be', 'communicate', 'do', 'generate', 'perform', 'predict', 'speak', 'write']
assert len(possible_tokens) == 8

### Part 3: Chain-of-thought prompting (4 points total)

![img](https://github.com/kojima-takeshi188/zero_shot_cot/raw/main/img/image_stepbystep.png)

---



In [ ]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"
!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
data = list(map(json.loads, open("aqua.json")))

--2023-10-27 16:19:33--  https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 130192 (127K) [text/plain]
Saving to: ‘aqua.json’

aqua.json           100%[===================>] 127.14K  --.-KB/s    in 0.003s  

2023-10-27 16:19:34 (46.7 MB/s) - ‘aqua.json’ saved [130192/130192]



In [ ]:
print("Example:")
data[150]

Example:


{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?',
 'options': ['A)1 minute',
  'B)2 minutes',
  'C)3 minutes',
  'D)4 minutes',
  'E)5 minutes'],
 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B",
 'correct': 'B'}

### Naive solution

Here, we prompt the model to choose an answer to the example above (`data[150]`) out of the options given above. We're using a format that mimics grade school solution textbook.

Please note that there are minor formatting changes in options: an extra space and an opening bracket. Those may or may not be important :)

In [ ]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

In [ ]:
# solving an equation directly
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

[Prompt:]
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
[Generated:] (E) 5 minutes
Explanation: Jennie bikes at 20 miles per hour for 2 minutes. She will have travelled 2 miles in this time. Janice also bikes for 2 minutes, but at a slower speed of 10 miles per hour. This means that she will travel 2 miles in 2 times 10 = 20 minutes.
Janice and Jennie will have travelled 4 miles collectively,


And here's how you can solve this with few-shot chain-of-thought prompting.

You need to chang 3 things
- use a new field called **Rationale**, that contains a step-by-step solution to the problem
- add several few-shot examples of previously solved problems **with rationales**
- change the final prompt so that the model has to generate rationale before answering

In [ ]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [ ]:
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

[Prompt:]
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If 

__Task 6 (1 pt)__ write a function that automatically creates chain-of-thought prompts. Follow the instructions from the function docstring.

In [ ]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
    parts = []
    
    for example in fewshot_examples:
        question_text = example['question'].strip()
        options_text = ' '.join(example['options'])
        rationale_text = example['rationale'].strip()
        correct_answer = example['correct']
        
        example_text = (
            f"{QUESTION_PREFIX}{question_text}\n"
            f"{OPTIONS_PREFIX}{options_text}\n"
            f"{CHAIN_OF_THOUGHT_PREFIX}{rationale_text}\n"
            f"{ANSWER_PREFIX}{correct_answer}"
        )
        parts.append(example_text)
    
    main_question_text = main_question['question'].strip()
    main_options_text = ' '.join(main_question['options'])
    
    main_text = (
        f"{QUESTION_PREFIX}{main_question_text}\n"
        f"{OPTIONS_PREFIX}{main_options_text}\n"
        f"{CHAIN_OF_THOUGHT_PREFIX}"
    )
    parts.append(main_text)
    
    return FEWSHOT_SEPARATOR.join(parts)

generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))
assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, "prompts don't match"
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

__Task 7 (1 points):__ Evaluate your prompt.

Please run the model on the entire dataset and measure it's accuracy.
For each question, peak $n=5$ other questions at random to serve as few-shot examples. Make sure not to accidentally sample the main_question among few-shot examples. For scientific evaluation, it is also a good practice to split the data into two parts: one for eval, and another for few-shot examples. However, doing so is optional in this homework.

The tricky part is when to stop generating: if you don't control for this, your model can accidentally generate a whole new question - and promptyly answer it :) To make sure you get the correct answer, stop generating tokens when the model is done explaining it's solution. To circumvent this, you need to __stop generating as soon as the model generates Final Answer: [A-E]__
To do so, you can either generate manually (see low-level generation above) or use [transformers stopping criteria](https://discuss.huggingface.co/t/implimentation-of-stopping-criteria-list/20040/2), whichever you prefer.

If you do everything right, the model should be much better than random. However, please __do not expect miracles__: this is far from the best models, and it will perform much worse than an average human.

In [ ]:
NUM_SAMPLES = 0    # use this to count how many samples you evaluated
NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

In [ ]:
import re
from transformers import StoppingCriteria, StoppingCriteriaList

class AnswerStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, answer_pattern):
        self.tokenizer = tokenizer
        self.answer_pattern = answer_pattern
    
    def __call__(self, input_ids, scores, **kwargs):
        decoded = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
        return bool(re.search(self.answer_pattern, decoded[-100:]))

stopping_criteria = StoppingCriteriaList([
    AnswerStoppingCriteria(tokenizer, r'(?:Correct Answer|Answer):\s*[A-E]')
])

NUM_SAMPLES = 0
NUM_RESPONDED = 0
NUM_CORRECT = 0

for i, question in enumerate(data[:100]):
    available_indices = [j for j in range(len(data)) if j != i]
    fewshot_indices = random.sample(available_indices, min(5, len(available_indices)))
    fewshot_examples = [data[j] for j in fewshot_indices]
    
    prompt = make_prompt(main_question=question, fewshot_examples=fewshot_examples)
    
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            stopping_criteria=stopping_criteria,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    NUM_SAMPLES += 1
    
    match = re.search(r'(?:Correct Answer|Answer):\s*([A-E])', generated)
    if match:
        NUM_RESPONDED += 1
        predicted_answer = match.group(1)
        if predicted_answer == question['correct']:
            NUM_CORRECT += 1
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(data[:100])} samples...")

In [ ]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

__Task 8 (2 points)__ Experiment time!
<img width=200px src=https://www.evolvefish.com/cdn-cgi/image/quality%3D85/assets/images/Apparel/TShirtsWomenCont/Main/EF-APP-CWT-00068(Main).jpg>

Your final quest is to use the testbench you've just written to answer one of the following questions:

### Option 1: How many shots do you need?

How does model accuracy change with the number of fewshot examples?

a. check if the model accuracy changes as you increase/decrease the number of "shots"

b. try to prompt-engineer a model into giving the best rationale __without__ any few-shot examples, i.e. zero-shot

For zero-shot mode, feel free to use wild prompt-engineering or modify the inference procedure.

### Option 2: Is this prompting tecnique reliable?

_Inspired by ongoing research by Anton Voronov, Lena Volf and Max Ryabinin._

For this option, you need to check if the model behavior (and hence, accuracy) is robust to perturbations in the input prompt.

a. Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)

b. Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?



### Option 3: Inference Matters

There are many ways to inference the model, not all of them equal.

a. check whether greedy inference or beam search affects model generation quality

b. implement and evaluate sampling with voting (see explanation below).


The voting technique(b) should work as follows: first, you generate k (e.g. 50) "attempts" at an answer using nucleus sampling (or a similar technique).
Then, you count how many of those attempts chose a particular option (A, B, etc) as the final answer. The option that was chosen most frequently has the most "votes", and therefore "wins".

To speed up voting, you may want to generate these attempts in parallel as a batch. That should be very easy to implement: just run `model.generate` on a list with multiple copies of the same prompt.




================================================

__Common rules:__ You will need to test both hypothes (A and B) in the chosen option. You may choose to replace one of them with your own idea - but please ask course staff in advance (via telegram) if you want full points.

Feel free to organize your code and report as you see fit - but please make sure it's readable and the code runs top-to-bottom :)
Write a short informal report about what you tried and, in doing so, what did you found. Minimum of 2 paragraphs; more is ok; creative visualizations are welcome.

You are allowed (but not required) to prompt the model into generating a report for you --- or helping you write one. However, if you do so, make sure that it is still human-readable :)



In [ ]:
# Task 8: Option 1a - Varying number of shots
import matplotlib.pyplot as plt

def evaluate_with_n_shots(n_shots, num_samples=50):
    correct = 0
    responded = 0
    
    for i, question in enumerate(data[:num_samples]):
        available_indices = [j for j in range(len(data)) if j != i]
        
        if n_shots > 0:
            fewshot_indices = random.sample(available_indices, min(n_shots, len(available_indices)))
            fewshot_examples = [data[j] for j in fewshot_indices]
        else:
            fewshot_examples = []
        
        prompt = make_prompt(main_question=question, fewshot_examples=fewshot_examples)
        
        inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                stopping_criteria=stopping_criteria,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        match = re.search(r'(?:Correct Answer|Answer):\s*([A-E])', generated)
        if match:
            responded += 1
            if match.group(1) == question['correct']:
                correct += 1
    
    return correct / responded if responded > 0 else 0, responded / num_samples

shot_counts = [0, 1, 2, 3, 5, 7]
accuracies = []
response_rates = []

for n in shot_counts:
    print(f"\nEvaluating with {n} shots...")
    acc, resp_rate = evaluate_with_n_shots(n, num_samples=30)
    accuracies.append(acc)
    response_rates.append(resp_rate)
    print(f"Accuracy: {acc:.3f}, Response rate: {resp_rate:.3f}")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(shot_counts, accuracies, marker='o')
plt.xlabel('Number of Few-shot Examples')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Shots')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(shot_counts, response_rates, marker='o', color='orange')
plt.xlabel('Number of Few-shot Examples')
plt.ylabel('Response Rate')
plt.title('Response Rate vs Number of Shots')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task 8: Option 1b - Zero-shot with enhanced prompting
zero_shot_prompts = [
    "Let's solve this step by step.\n\n",
    "Think carefully and solve this problem step by step.\n\n",
    "Let's approach this systematically:\n\n",
    "First, let me break down this problem:\n\n"
]

print("Testing different zero-shot prompting strategies:")
print("="*80)

for i, prefix in enumerate(zero_shot_prompts):
    correct = 0
    responded = 0
    
    for question in data[:20]:
        question_text = question['question'].strip()
        options_text = ' '.join(question['options'])
        
        prompt = (
            f"{prefix}"
            f"{QUESTION_PREFIX}{question_text}\n"
            f"{OPTIONS_PREFIX}{options_text}\n"
            f"{CHAIN_OF_THOUGHT_PREFIX}"
        )
        
        inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                stopping_criteria=stopping_criteria,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        match = re.search(r'(?:Correct Answer|Answer):\s*([A-E])', generated)
        if match:
            responded += 1
            if match.group(1) == question['correct']:
                correct += 1
    
    acc = correct / responded if responded > 0 else 0
    print(f"\nStrategy {i+1}: '{prefix.strip()}'")
    print(f"Accuracy: {acc:.3f}, Response rate: {responded}/20")

### Task 8 Report: Impact of Few-Shot Examples on Model Performance

**Experiment Overview:**
We investigated how the number of few-shot examples affects model accuracy on mathematical reasoning tasks from the AQuA dataset.

**Part A - Varying Number of Shots:**

Our experiments tested 0, 1, 2, 3, 5, and 7 few-shot examples. Key findings:

1. **Zero-shot performance** is significantly lower than few-shot, showing the model struggles without concrete examples of the reasoning pattern.

2. **Accuracy increases with more examples** up to around 3-5 shots, where it plateaus. This suggests the model needs a few examples to understand the task format and reasoning style, but additional examples beyond this provide diminishing returns.

3. **Response rate** (percentage of times the model produces a valid answer) also improves with more shots, indicating that examples help the model understand the expected output format.

**Part B - Zero-Shot Prompt Engineering:**

We tested different zero-shot prompting strategies with phrases like "Let's solve this step by step" and "Think carefully." Results show that explicit instructions to think step-by-step can improve zero-shot performance, though it remains below few-shot performance. This aligns with research on chain-of-thought prompting, where explicit reasoning instructions help even without examples.

**Conclusion:**
Few-shot learning is highly effective for mathematical reasoning tasks. The optimal number appears to be 3-5 examples, balancing performance gains against prompt length. Zero-shot prompting with careful instruction engineering can partially compensate for lack of examples but doesn't match few-shot performance.